# Retrieve relevant NASA clauses

For each of the 54 SRS requirements, this notebook retrieves the most semantically 
relevant NASA clauses from the vector database built by 03_build_vector_store.ipynb.

The results are saved to data/rag_retrievals.csv for use in compliance classification.

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# find the repo root by looking for the data/ folder, so this works from any directory
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "srs_requirements.csv").exists())
data_dir = ROOT / "data"

srs_df = pd.read_csv(data_dir / "srs_requirements.csv")
print(srs_df.shape)
print(srs_df.columns.tolist())

(54, 4)
['req_id', 'title', 'text', 'section']


In [3]:
srs_df.head(3)

,req_id,title,text,section
0,SFMC-REQ-001,Position Display,The SFMC shall display the spacecraft's curren...,3.1 Navigation
1,SFMC-REQ-002,Navigation Update,The SFMC shall update the displayed spacecraft...,3.1 Navigation
2,SFMC-REQ-003,Destination Selection,The SFMC shall allow the pilot to select a nav...,3.1 Navigation


In [4]:
print(srs_df.iloc[0]['text'])

The SFMC shall display the spacecraft's current position to the pilot.


In [5]:
clauses_df = pd.read_csv(data_dir / "addressable_clauses.csv")
print(clauses_df.shape)
print(clauses_df.columns.tolist())

(62, 10)
['chunk_id', 'swe_id', 'section', 'text', 'source', 'source_url', 'type', 'applies_to', 'interpretation', 'review_note']


In [6]:
print(clauses_df.iloc[0]['text'])

If a project has safety-critical software or mission-critical software, the project manager shall implement the following items in the software:  a. The software is initialized, at first start and restarts, to a known safe state. b. The software safely transitions between all predefined known states. c. Termination performed by software functions is performed to a known safe state. d. Operator overrides of software functions require at least two independent actions by an operator. e. Software rejects commands received out of sequence when execution of those commands out of sequence can cause a hazard. f. The software detects inadvertent memory modification and recovers to a known safe state. g. The software performs integrity checks on inputs and outputs to/from the software system. h. The software performs prerequisite checks prior to the execution of safety-critical software commands. i. No single software event or action is allowed to initiate an identified hazard. j. The software r

## Load the vector store

Load the Chroma database that Victor built in 03_build_vector_store.ipynb.
Make sure you have run that notebook first — it creates data/chroma_db/ locally.

In [7]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embedding_fn = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma(
    collection_name="addressable_clauses",
    persist_directory=str(data_dir / "chroma_db"),
    embedding_function=embedding_fn,
)

print(vectorstore._collection.count())

164


## Retrieval function

Searches the vector store for the most relevant NASA clauses for a given query.
We fetch more results than needed (k=50) and then deduplicate by chunk_id, because 
long clauses get split into multiple pieces — we want distinct clauses, not duplicate pieces.

In [8]:
def retrieve(query, k=5):
    results = []
    seen = set()
    
    # fetch more than k so we still get k after deduplication
    raw = vectorstore.similarity_search_with_score(query, k=50)
    
    for doc, score in raw:
        chunk_id = doc.metadata['chunk_id']
        if chunk_id not in seen:
            seen.add(chunk_id)
            results.append((doc, score))
        if len(results) == k:
            break
    
    return results

## Quick test

Test the retrieval function on one requirement before running it on all 54.

In [9]:
# test it on REQ-001
for doc, score in retrieve(srs_df.iloc[0]['text']):
    print(f"{score:.3f}  {doc.metadata['chunk_id']}")

1.433  swehb-6.1-guidance
1.438  swehb-3.1.2-guidance
1.486  swehb-3.6-guidance
1.497  swehb-3.1.1-guidance
1.552  swehb-7.1-guidance


## Experimenting with k

k controls how many NASA clauses we retrieve per requirement. 
Too small and we might miss relevant clauses. Too large and we start getting irrelevant ones.
We test k=3, 5, and 10 on the fault log requirement (SFMC-REQ-029) since we know it has clear matches.

In [10]:
test_req = srs_df.loc[srs_df['req_id'] == 'SFMC-REQ-029', 'text'].item()

for k in [3, 5, 10]:
    print(f"--- k={k} ---")
    for doc, score in retrieve(test_req, k=k):
        print(f"  {score:.3f}  {doc.metadata['chunk_id']}")
    print()

--- k=3 ---
  1.003  swehb-3.2.1.9-guidance
  1.015  swehb-6.1-guidance
  1.115  swehb-3.2.1.13-guidance

--- k=5 ---
  1.003  swehb-3.2.1.9-guidance
  1.015  swehb-6.1-guidance
  1.115  swehb-3.2.1.13-guidance
  1.139  swehb-3.2.1.4-guidance
  1.177  swehb-7.1-guidance

--- k=10 ---
  1.003  swehb-3.2.1.9-guidance
  1.015  swehb-6.1-guidance
  1.115  swehb-3.2.1.13-guidance
  1.139  swehb-3.2.1.4-guidance
  1.177  swehb-7.1-guidance
  1.195  npr-3.7.3-requirement
  1.229  swehb-3.2.1.2-guidance
  1.243  swehb-1.b-guidance
  1.248  npr-4.1.4-requirement
  1.249  swehb-3.1.3-guidance



## Run retrieval for all requirements

We use k=5 based on the experiment above — scores stabilise after the top 3 
and don't drop sharply until around rank 8-9. k=5 gives a small buffer without 
pulling in clearly irrelevant clauses.

In [11]:
K = 5

all_results = []

for _, row in srs_df.iterrows():
    for rank, (doc, score) in enumerate(retrieve(row['text'], k=K), start=1):
        all_results.append({
            'req_id':   row['req_id'],
            'rank':     rank,
            'chunk_id': doc.metadata['chunk_id'],
            'swe_id':   doc.metadata['swe_id'],
            'score':    round(score, 3),
        })

results_df = pd.DataFrame(all_results)
print(results_df.shape)
results_df.head(10)

(270, 5)


,req_id,rank,chunk_id,swe_id,score
0,SFMC-REQ-001,1,swehb-6.1-guidance,,1.433
1,SFMC-REQ-001,2,swehb-3.1.2-guidance,,1.438
2,SFMC-REQ-001,3,swehb-3.6-guidance,,1.486
3,SFMC-REQ-001,4,swehb-3.1.1-guidance,,1.497
4,SFMC-REQ-001,5,swehb-7.1-guidance,,1.552
5,SFMC-REQ-002,1,swehb-3.6-guidance,,1.381
6,SFMC-REQ-002,2,swehb-3.2.1.5-guidance,,1.479
7,SFMC-REQ-002,3,swehb-3.2.1.2-guidance,,1.500
8,SFMC-REQ-002,4,swehb-3.1.2-guidance,,1.528
9,SFMC-REQ-002,5,npr-4.1.6-requirement,SWE-054,1.551


## Save results

Save the retrieval results to data/rag_retrievals.csv for use in the next stage 
of the pipeline (compliance classification).

In [12]:
results_df.to_csv(data_dir / "rag_retrievals.csv", index=False)
print(f"Saved {len(results_df)} rows to data/rag_retrievals.csv")

Saved 270 rows to data/rag_retrievals.csv
